<a href="https://colab.research.google.com/github/TrKhuyn/K4-Track4-Day1-Neuralnetwork/blob/main/code/lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Neural Network trên Google Colab
Chọn **Runtime → Change runtime type → T4 GPU**, điền MSSV và chạy **Run all**. Không dùng eval để tuning.

In [1]:
# Nếu mở notebook bằng link GitHub, Colab cần clone repo trước.
REPO_URL='https://github.com/TrKhuyn/K4-Track4-Day1-Neuralnetwork.git'
MSSV='2A202602682'
assert REPO_URL.startswith('http') and MSSV!='2A202602682'
import os,sys,subprocess,shutil,json,math
from pathlib import Path
if not Path('/content/lab1').exists(): subprocess.run(['git','clone',REPO_URL,'/content/lab1'],check=True)
os.chdir('/content/lab1/code'); REPO_ROOT=Path('/content/lab1')
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive')/f'submission_{MSSV}'
for d in ('code','figures','results'): (OUT/d).mkdir(parents=True,exist_ok=True)
for p in Path('.').glob('*.py'): shutil.copy2(p,OUT/'code'/p.name)
import numpy as np, torch
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.__version__,device,torch.cuda.get_device_name(0) if device.type=='cuda' else '')
from data import prepare_data
from model import MLP,EXPECTED_PARAMS,count_params,activation_stats
from train import DEFAULT_CFG,evaluate,run_experiment,final_eval,compute_loss
from plots import plot_run,plot_compare
from results_table import save_result,to_row,write_xlsx


AssertionError: 

## Part 0–1 — Dữ liệu và sanity checks
Validation được tách phân tầng từ train; standardizer chỉ fit trên training subset.

In [ ]:
processed=REPO_ROOT/'data'/'processed'
if not (processed/'train.npz').exists(): subprocess.run([sys.executable,'scripts/split_data.py'],cwd=REPO_ROOT,check=True)
data=prepare_data(str(device),0.2,42,str(processed))
model=MLP(init='he').to(device); assert count_params(model)==EXPECTED_PARAMS[(256,128)]
assert model(torch.randn(8,54,device=device)).shape==(8,7)
print('step0 / ln7:',evaluate(model,data['X_val'],data['y_val'])['loss'],math.log(7))
print('activation std:',activation_stats(model,data['X_val'][:512]))
tiny=MLP(init='he').to(device); opt=torch.optim.Adam(tiny.parameters(),lr=1e-2); losses=[]
for _ in range(400):
    opt.zero_grad(set_to_none=True); loss=compute_loss(tiny(data['X_tr'][:20]),data['y_tr'][:20],'ce'); loss.backward(); opt.step(); losses.append(loss.item())
print('overfit 20:',losses[0],losses[-1]); assert losses[-1]<.05


## Part 2–3 — Baseline, seed noise và thí nghiệm
Mỗi thí nghiệm đổi một yếu tố, cùng split và 20 epoch. Hãy ghi dự đoán trước và nhận xét sau vào báo cáo.

In [ ]:
def run_save(cfg):
    r=run_experiment(cfg,data); save_result(r,str(OUT/'results')); plot_run(r,str(OUT/'figures'/f"{cfg['exp_id']}.png")); return r
base={**DEFAULT_CFG,'lr':.05,'epochs':20}
bases=[run_save({**base,'seed':s,'exp_id':f'base-s{s}'}) for s in (1,2,3)]
f=np.array([r['summary']['val_macro_f1'] for r in bases]); print('baseline mean/std/2sigma:',f.mean(),f.std(ddof=1),2*f.std(ddof=1))
cfgs=[
 {**base,'exp_id':'opt-adam','group':'optimizer','description':'Adam','optimizer':'adam','lr':1e-3},
 {**base,'exp_id':'opt-adamw','group':'optimizer','description':'AdamW','optimizer':'adamw','lr':1e-3,'weight_decay':1e-4},
 {**base,'exp_id':'dropout-02','group':'dropout','description':'Dropout .2','dropout':.2},
 {**base,'exp_id':'clip-10','group':'clipping','description':'Clip norm 1','clip_norm':1.},
 {**base,'exp_id':'loss-mse','group':'loss','description':'MSE one-hot','loss':'mse'},
 {**base,'exp_id':'init-xavier','group':'init','description':'Xavier','init':'xavier'}]
others=[run_save(c) for c in cfgs]; results=bases+others
for group in ('optimizer','dropout','clipping','loss','init'):
    rs=[bases[0]]+[r for r in others if r['cfg']['group']==group]; plot_compare(rs,'val_macro_f1',str(OUT/'figures'/f'compare_{group}.png'),group)
[(r['cfg']['exp_id'],round(r['summary']['val_macro_f1'],4),r['summary']['best_epoch']) for r in results]


## Part 4 — Chọn bằng validation và đánh giá eval
Cấu hình cuối được chọn trước khi đọc điểm eval.

In [ ]:
winner=max(results,key=lambda r:r['summary']['val_macro_f1']); print('selected by val:',winner['cfg']['exp_id'])
pred=OUT/'predictions_eval.csv'; final_eval(winner['cfg'],winner,data,str(pred))
ev=OUT/'eval_result.json'; subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(pred),'--out',str(ev)],cwd=REPO_ROOT,check=True)
scores=json.loads(ev.read_text()); print('final eval:',scores['accuracy'],scores['macro_f1'])
bp=OUT/'predictions_baseline_eval.csv'; final_eval(bases[0]['cfg'],bases[0],data,str(bp))
be=OUT/'eval_result_baseline.json'; subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(bp),'--out',str(be)],cwd=REPO_ROOT,check=True)
bs=json.loads(be.read_text()); print('baseline eval:',bs['accuracy'],bs['macro_f1'])
rows=[to_row(r,scores if r is winner else (bs if r is bases[0] else None)) for r in results]
write_xlsx(rows,str(REPO_ROOT/'templates'/'experiment_table_template.xlsx'),str(OUT/'experiments.xlsx'))
shutil.copy2(REPO_ROOT/'templates'/'REPORT_TEMPLATE.md',OUT/'REPORT.md'); shutil.copy2(REPO_ROOT/'code'/'lab.ipynb',OUT/'code'/'lab.ipynb')
print('Output:',OUT); print('Viết REPORT.md và nhận xét dựa trên số liệu thật trước khi nộp.')
